# Example 6a: 阶段分离 — 前处理 + 求解（不做提取）

演示 `BatchAbaqusProcessor` 的阶段分离能力：`run_preparation()` / `run_simulation()` / `run_extraction()`
可以分开独立调用，不必每次都走完整的 `run_batch()` 流程。

这个文件只做**前处理**和**求解**两个阶段，刻意不做任何结果提取 —— 后处理提取被放在
另一个独立的文件 `02_extract_odb.ipynb` 里，模拟"先把 job 跑完，过一段时间/换一个脚本
再回来提取结果"的真实场景。

In [6]:
import os

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()
OUTPUT_DIR = os.path.join(CWD, "examples/06_SeparateJob/output")

## 构造 3 个不同弹性模量的 Job

`job_name` 和 `base_output_dir` 需要在 `02_extract_odb.ipynb` 里原样复用，
这样 `BatchAbaqusProcessor._build_calc()` 才能推导出同一个 `output_dir` 找到已有的 ODB。

In [7]:
YOUNGS_MODULUS_LIST = [190000, 200000, 210000]

specs = [
	JobSpec(
		job_name = f"separate_job_{i:02d}",
		workflow = "modular",
		preparation = PreparationSpec(
			kind = "inp_based",
			source_path = "./examples/cae_file/planar_stress_template.inp",
			params = {
				"youngs_modulus": e,
				"load_magnitude": 2000,
			}
		),
		post_extraction = [
			HookSpec(
				script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
				tasks = [
					{"result_name": "max_stress_mises",},
					{"result_name": "max_displacement",},
				]
			)
		]
	)
	for i, e in enumerate(YOUNGS_MODULUS_LIST, start=1)
]

In [8]:
processor = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = OUTPUT_DIR,
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

## 阶段 1：只跑前处理（`run_preparation`）

只生成 INP，不求解、不提取。默认 `num_parallel_jobs=1`，可按需调大。

In [9]:
prep_outcomes = processor.run_preparation(num_parallel_jobs=3)

for oc in prep_outcomes:
	inp_path = os.path.join(oc.output_dir, f"{oc.job_name}.inp")
	print(oc.job_name, oc.status, "inp exists:", os.path.isfile(inp_path))

Output()

separate_job_01 COMPLETED inp exists: True
separate_job_02 COMPLETED inp exists: True
separate_job_03 COMPLETED inp exists: True


## 阶段 2：只跑求解（`run_simulation`）

假设 `run_preparation()` 已经产出了 INP（存在于 `ctx.inp_path`）。
`run_simulation()` 内部会先跑 `pre_extraction`（这里没有配置，跳过），再提交求解器。
注意这里 `oc.results` 是空字典 —— `post_extraction` 属于提取阶段，不在这一步执行。

In [10]:
sim_outcomes = processor.run_simulation(num_parallel_jobs=3)

for oc in sim_outcomes:
	odb_path = os.path.join(oc.output_dir, f"{oc.job_name}.odb")
	print(oc.job_name, oc.status, f"{oc.duration_s:.1f}s", "odb exists:", os.path.isfile(odb_path))
	print("  results (应为空，没有 pre_extraction 钩子):", oc.results)
	print("  phases:", [p["phase"] for p in (oc.phases or [])])

Output()

separate_job_03 COMPLETED 45.6s odb exists: True
  results (应为空，没有 pre_extraction 钩子): {}
  phases: ['simulation']
separate_job_02 COMPLETED 45.7s odb exists: True
  results (应为空，没有 pre_extraction 钩子): {}
  phases: ['simulation']
separate_job_01 COMPLETED 48.3s odb exists: True
  results (应为空，没有 pre_extraction 钩子): {}
  phases: ['simulation']


## 到此为止

三个 job 都已经求解完毕（ODB 已经写好），但还没有做任何结果提取。
打开 `02_extract_odb.ipynb`，在一个全新的 Python/Jupyter 会话里独立完成提取。